# Classification

The goal for this notebook is to predict how long will it take to do a recipe based on the [Recipe NLG Dataset](https://recipenlg.cs.put.poznan.pl/)

## ⬇️ Imports

In [1]:
import pandas as pd
import ast
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
import spacy
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

ps = PorterStemmer()

## 🛠️ Preprocessing
### Downloading the dataset

If not already done use this [notebook](../../preprocessing/preprocessing.ipynb) to download the dataset in your local directory.

In [8]:
data = pd.read_csv('../../../data/processed/preprocessed_recipe.csv')

In [9]:
data.head()

,id,name,minutes,n_steps,description,n_ingredients,steps_string_standardize,ingredients_text,tags_text,calories,total_fat,sugar,sodium,protein,saturated_fat,carbohydrates
0,137739,arriba baked squash mexican,55,11,autumn is my favorite time of year to cook! th...,7,make a choic and proceed with recip depend on ...,"['winter squash', 'mexican seasoning', 'mixed ...","['60-minutes-or-less', 'time-to-make', 'course...",51.5,0.0,13.0,0.0,2.0,0.0,4.0
1,31490,breakfast pizza,30,9,this recipe calls for the crust to be prebaked...,6,preheat oven to 103.33 celsius °c press dough ...,"['prepared pizza crust', 'sausage patty', 'egg...","['30-minutes-or-less', 'time-to-make', 'course...",173.4,18.0,0.0,17.0,22.0,35.0,1.0
2,112140,chili,130,6,this modified version of 'mom's' chili was a h...,13,brown ground beef in larg pot add chop onion t...,"['ground beef', 'yellow onions', 'diced tomato...","['time-to-make', 'course', 'preparation', 'mai...",269.8,22.0,32.0,48.0,39.0,27.0,5.0
3,59389,alouette potato,45,11,"this is a super easy, great tasting, make ahea...",11,place potato in a larg pot of lightli salt wat...,"['spreadable cheese with garlic and herbs', 'n...","['60-minutes-or-less', 'time-to-make', 'course...",368.1,17.0,10.0,2.0,14.0,8.0,20.0
4,44061,amish tomato ketchup canning,190,5,my dh's amish mother raised him on this recipe...,8,"mix all ingredients& boil for 2 30.0 minute , ...","['tomato juice', 'apple cider vinegar', 'sugar...","['weeknight', 'time-to-make', 'course', 'main-...",352.9,1.0,337.0,23.0,3.0,0.0,28.0


In [104]:
""" seems to have no effect
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "tagger", "lemmatizer"])
texts = data["steps_string_standardize"].astype(str).tolist()
num_cpus = os.cpu_count()
n_process = max(1, int(num_cpus * 0.7)) if num_cpus else 1
docs = list(nlp.pipe(texts, batch_size=50, n_process=n_process))
data["steps_tokens"] = [[token.text for token in doc] for doc in docs]
"""

' seems to have no effect\nnlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "tagger", "lemmatizer"])\ntexts = data["steps_string_standardize"].astype(str).tolist()\nnum_cpus = os.cpu_count()\nn_process = max(1, int(num_cpus * 0.7)) if num_cpus else 1\ndocs = list(nlp.pipe(texts, batch_size=50, n_process=n_process))\ndata["steps_tokens"] = [[token.text for token in doc] for doc in docs]\n'

In [15]:
nan_rows = data[pd.isna(data['steps_string_standardize'])]
nan_rows.head()

,id,name,minutes,n_steps,description,n_ingredients,steps_string_standardize,ingredients_text,tags_text,calories,total_fat,sugar,sodium,protein,saturated_fat,carbohydrates
3160,176767,bread,90,0,just change the fruit/vegetable in this recipe...,12,NaN,"['flour', 'baking soda', 'salt', 'baking powde...","['time-to-make', 'course', 'main-ingredient', ...",198.8,11.0,70.0,18.0,5.0,5.0,10.0


In [11]:
steps_vectorizer = TfidfVectorizer(max_features=500, stop_words='english')
steps_features = steps_vectorizer.fit_transform(data["steps_string_standardize"])

ingredients_vectorizer = TfidfVectorizer(max_features=300, stop_words='english')
ingredients_features = ingredients_vectorizer.fit_transform(data["ingredients_text"])

tags_vectorizer = TfidfVectorizer(max_features=300, stop_words='english')
tags_features = ingredients_vectorizer.fit_transform(data["tags_text"])

ValueError: np.nan is an invalid document, expected byte or unicode string.

In [11]:
#numerical_features = data[['n_steps', 'n_ingredients', 'token_count', 'avg_token_length', 'calories', 'total_fat', 'sugar', 'sodium', 'saturated_fat', 'carbohydrates']].values
numerical_features = data[['n_steps', 'n_ingredients', 'calories', 'total_fat', 'sugar', 'sodium', 'saturated_fat', 'carbohydrates']].values

feature_matrices = [steps_features.toarray(), ingredients_features.toarray(),
                    tags_features.toarray(),
                   numerical_features]

In [12]:
X = np.hstack(feature_matrices)
y = data["minutes"].values

In [13]:
X_train, X_test, y_train, y_test, X_train_indices, X_test_indices = train_test_split(
    X, y, data.index, test_size=0.2, random_state=42
)

## Training our model and Defining Metrics


- MAE :
    - Represents how much is it off on average
    - Ex : You make predictions that are off by 15 minutes on average

- R^2 :
    - Ranges from negative to 1
    - The closer it is to 1 the better it is
    - Under 0 it performs worse then random

In [14]:
def get_metrics(y_test, y_pred):
    # MAE
    print("MAE:", mean_absolute_error(y_test, y_pred))

    # R²
    print("R²:", r2_score(y_test, y_pred))

print("Linear Regression Results:")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
lr_metrics = get_metrics(y_test, y_pred_lr)

Linear Regression Results:
MAE: 14.620262500091304
R²: 0.7096547018831727


In [15]:
def predict_cooking_time(steps_text, ingredients_list, n_steps, n_ingredients, nutrition_values=None, tags=None):
    """
    Predict cooking time for a new recipe
    
    Parameters:
    steps_text (str): The recipe steps as a single string
    ingredients_list (list): List of ingredients
    n_steps (int): Number of steps in the recipe
    n_ingredients (int): Number of ingredients
    nutrition_values (dict, optional): Dictionary containing nutrition values with keys:
                     'calories', 'total_fat', 'sugar', 'sodium', 'protein', 'saturated_fat', 'carbohydrates'
    tags (list, optional): List of recipe tags/categories
    
    Returns:
    float: Predicted cooking time in minutes
    """
    # Process steps text
    standardized_steps = standardize_units(steps_text)
    #doc = nlp(standardized_steps)
    #steps_tokens = [token.text for token in doc]
    #token_count = len(steps_tokens)
    #avg_token_length = np.mean([len(token) for token in steps_tokens]) if steps_tokens else 0
    
    # Create ingredients text
    ingredients_text = ' '.join(ingredients_list)
    
    # Create tags text - but we'll use ingredients_vectorizer as that's what was used in training
    tags_text = ' '.join(tags) if tags else ''
    
    # Transform text features
    steps_features = steps_vectorizer.transform([standardized_steps]).toarray()
    ingredients_features = ingredients_vectorizer.transform([ingredients_text]).toarray()
    tags_features = ingredients_vectorizer.transform([tags_text]).toarray() if tags else np.zeros((1, 300))
    
    # Default nutrition values if not provided
    if nutrition_values is None:
        nutrition_values = {
            'calories': 0,
            'total_fat': 0,
            'sugar': 0,
            'sodium': 0,
            'protein': 0,
            'saturated_fat': 0,
            'carbohydrates': 0
        }
    
    # Combine numerical features - must match the order used in training
    numerical_features = np.array([[
        n_steps, 
        n_ingredients, 
        #token_count, 
        #avg_token_length,
        nutrition_values['calories'],
        nutrition_values['total_fat'],
        nutrition_values['sugar'],
        nutrition_values['sodium'],
        nutrition_values['saturated_fat'],
        nutrition_values['carbohydrates']
    ]])
    
    # Stack all features - ensure this matches the training stack order
    features = np.hstack([steps_features, ingredients_features, tags_features, numerical_features])

    # Make prediction
    prediction = lr_model.predict(features)[0]
    
    return max(1, prediction)  # Ensure prediction is at least 1 minute


for i in range(4):
    row = data.iloc[X_test_indices[i]]  # X_test_indices will be defined below
    predicted_time = predict_cooking_time(
        row['steps_string_standardize'],
        row['ingredients'],
        row['n_steps'],
        row['n_ingredients'],
        {
            'calories': row['calories'],
            'total_fat': row['total_fat'],
            'sugar': row['sugar'],
            'sodium': row['sodium'],
            'protein': row['protein'],
            'saturated_fat': row['saturated_fat'],
            'carbohydrates': row['carbohydrates']
        },
        row['tags']
    )
    print(f"\nTest n°{i} : Predicted cooking time for '{row['name']}: {predicted_time:.1f} minutes")
    print(f"Test n°{i} : Actual cooking time: {row['minutes']:.1f} minutes")

# Test n°2 without tags
sample_steps = "Preheat oven to 350°F. Mix ingredients in a bowl. Bake for 30 minutes."
sample_ingredients = ["flour", "sugar", "eggs", "butter"]
sample_n_steps = 3
sample_n_ingredients = len(sample_ingredients)
sample_nutrition = {
    'calories': 350,
    'total_fat': 15,
    'sugar': 20,
    'sodium': 200,
    'protein': 5,
    'saturated_fat': 8,
    'carbohydrates': 45
}

predicted_time = predict_cooking_time(
    sample_steps, 
    sample_ingredients, 
    sample_n_steps, 
    sample_n_ingredients,
    sample_nutrition
)

print(f"\nTest n°2 : Predicted cooking time for random recipe: {predicted_time:.1f} minutes")
print(f"Test n°2 : No actual cooking time registered")


Test n°0 : Predicted cooking time for 'creamy santa fe chicken rice: 76.9 minutes
Test n°0 : Actual cooking time: 75.0 minutes

Test n°1 : Predicted cooking time for 'pommes: 97.0 minutes
Test n°1 : Actual cooking time: 125.0 minutes

Test n°2 : Predicted cooking time for 'crunchy oriental pork casserole: 38.6 minutes
Test n°2 : Actual cooking time: 35.0 minutes

Test n°3 : Predicted cooking time for 'southwestern onion ring: 47.6 minutes
Test n°3 : Actual cooking time: 60.0 minutes

Test n°2 : Predicted cooking time for random recipe: 68.9 minutes
Test n°2 : No actual cooking time registered
